In [1]:
import os
import glob
import numpy as np
import xarray as xr
import pandas as pd
import dask
import dask.array as da
from datetime import timezone, timedelta

from dask import delayed, compute
from tqdm import tqdm
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar
from dask.distributed import wait

import time

# ============================
# User settings
# ============================

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")

# BARRA-C2 variable paths
u_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/ua100m/latest/"
v_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/va100m/latest/"


In [2]:
client = Client(n_workers=18,
    threads_per_worker=1,
    memory_limit=f"{int(7)}GB"
)

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 18
Total threads: 18,Total memory: 117.35 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41973,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38463,Total threads: 1
Dashboard: /proxy/36547/status,Memory: 6.52 GiB
Nanny: tcp://127.0.0.1:37459,


2026-05-19 13:38:07,132 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 30ae31a67887a6d2bb0be626e9d8bbf4 initialized by task ('rechunk-merge-rechunk-transfer-eb78ad1f5604f365692a779ad6525211', 0, 0, 0, 0, 0, 0, 0, 0) executed on worker tcp://127.0.0.1:46407
2026-05-19 13:38:07,869 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 057a9fa5d5fc15b8b8a2d5f373d00186 initialized by task ('rechunk-merge-rechunk-transfer-25d256d98ae6d4bfc27fd55742a74909', 0, 0, 0, 0, 0, 0, 0, 0) executed on worker tcp://127.0.0.1:34153
2026-05-19 13:38:08,888 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 6744ace3d543e26fde47879edcb124e3 initialized by task ('rechunk-merge-rechunk-transfer-ab710db1fe75bfabc01cf615f8c5f959', 0, 0, 33, 0, 0, 0, 135, 0) executed on worker tcp://127.0.0.1:40155
2026-05-19 13:38:08,908 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 6a5a42ac54039c9c1cae76f25300adaf initialized by task ('rechunk-merge-rechunk-transfer-cca91668dcae376fc

These define whether which sample we're calcualting and which model we're using.

In [3]:
reanalysis = 'BARRA-C2'

In [4]:
hw_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_cluster.csv")
bl_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/no_hw_alpine_cluster.csv")

if reanalysis == 'BARRA-C2':
    # Alpine NSW extent
    # extent = [147, 152, -38.5, -32]
    # # South Australia extent
    extent = [137, 142, -36, -31,]

elif reanalysis == 'BARRA-R2':
    extent = None

lon_min, lon_max, lat_min, lat_max = extent

In [5]:
cluster = hw_dates.columns.tolist()
cluster.pop(0)

cluster = gen_csv[gen_csv['DUID'].isin(cluster)][['DUID','lat','lon']]
cluster

,DUID,lat,lon
36,BLUFF1,-33.369331,138.982476
89,CLEMGPWF,-33.508568,138.119175
185,HALLWF1,-33.348652,138.751659
186,HALLWF2,-33.348652,138.751659
190,HDWF1,-33.042878,138.556285
191,HDWF2,-33.042878,138.556285
192,HDWF3,-33.042839,138.556216
240,LGAPWF1,-32.564508,137.557192
241,LGAPWF2,-32.564508,137.557192
310,NBHWF1,-33.252576,138.712289


In [6]:
def get_days(days_df, nc_dir, tz='Etc/GMT-10', extent=None):
    """
    Load all hours for specified local calendar dates.
    Adds local_hour (0-23) and local_day (naive local midnight) coordinates.
    """
    if extent is None:
        raise ValueError("extent must be provided: [lon_min, lon_max, lat_min, lat_max]")
    if 'date' not in days_df:
        raise KeyError("days_df must contain 'date' column")

    # Parse & de-duplicate dates (assumed day strings)
    ser = pd.to_datetime(days_df['date'], errors='coerce').dropna().drop_duplicates()
    if ser.empty:
        raise ValueError("No valid dates after parsing.")
    local_dates = pd.DatetimeIndex(ser).tz_localize(tz)

    # UTC coverage window
    utc_start = local_dates.min().tz_convert('UTC').floor('D')
    utc_end   = (local_dates.max() + pd.Timedelta(days=1)).tz_convert('UTC').ceil('D')

    # Month filter
    month_filter = set(pd.date_range(
        utc_start, utc_end - pd.Timedelta(seconds=1), freq='MS', tz='UTC'
    ).strftime('%Y%m'))

    files = sorted(
        f for f in glob.glob(os.path.join(nc_dir, "*"))
        if any(ym in os.path.basename(f) for ym in month_filter)
    )
    if not files:
        raise FileNotFoundError(f"No files for months {month_filter} in {nc_dir}")

    lon_min, lon_max, lat_min, lat_max = extent

    def _pre(ds):
        return ds.sel(
            lon=slice(lon_min, lon_max),
            lat=slice(lat_min, lat_max),
            time=slice(utc_start.tz_localize(None), utc_end.tz_localize(None))
        )

    ds = xr.open_mfdataset(
        files,
        preprocess=_pre,
        combine='by_coords',
        parallel=True,
        chunks='auto',
        engine='netcdf4',
        compat='no_conflicts'
    )

    # Localize times
    utc_index = ds.indexes['time']
    local_index = utc_index.tz_localize('UTC').tz_convert(tz)

    # Mask to requested local days
    wanted = set(local_dates.normalize())
    mask = pd.Index(local_index.normalize()).isin(wanted)

    ds = ds.isel(time=mask)

    # Build coords
    local_hour_vals = local_index[mask].hour.astype('int32')
    # Strip time zone after normalizing so we retain local midnight as naive
    local_day_vals = local_index[mask].normalize().tz_localize(None)

    ds = ds.assign_coords(
        local_hour=('time', local_hour_vals),
        local_day =('time', local_day_vals)
    )
    return ds

def load_uv_for_dates(days_df, u_path, v_path, tz='Etc/GMT-10', extent=None):
    du = get_days(days_df, u_path, tz=tz, extent=extent)
    dv = get_days(days_df, v_path, tz=tz, extent=extent)
    return xr.merge([du, dv])

In [7]:
dates_series = pd.concat([hw_dates['date'], bl_dates['date']])
all_dates_df = (
    pd.to_datetime(dates_series, errors='coerce')
      .dropna()
      .drop_duplicates()
      .sort_values()
      .to_frame(name='date')
    )

In [8]:
%%time
ds_all = load_uv_for_dates(
    all_dates_df,
    u_path,
    v_path,
    tz='Etc/GMT-10',
    extent=extent).persist()

wait(ds_all)

/jobfs/168778698.gadi-pbs/ipykernel_164796/2316493325.py:76: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  return xr.merge([du, dv])
/jobfs/168778698.gadi-pbs/ipykernel_164796/2316493325.py:76: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  return xr.merge([du, dv])
/jobfs/168778698.gadi-pbs/ipykernel_164796/2316493325.py:76: FutureWar

CPU times: user 1min 1s, sys: 12.3 s, total: 1min 14s
Wall time: 1min 53s


DoneAndNotDoneFutures(done={<Future: finished, type: numpy.ndarray, key: ('getitem-19933a3497a934b3e572499345f54460', 2213, 0, 0)>, <Future: finished, type: numpy.ndarray, key: ('getitem-6fa12382634b1c8113b3fe2ec652ff61', 297, 0, 0)>, <Future: finished, type: numpy.ndarray, key: ('getitem-6fa12382634b1c8113b3fe2ec652ff61', 1456, 0, 0)>, <Future: finished, type: numpy.ndarray, key: ('getitem-6fa12382634b1c8113b3fe2ec652ff61', 575, 0, 0)>, <Future: finished, type: numpy.ndarray, key: ('getitem-6fa12382634b1c8113b3fe2ec652ff61', 155, 0, 0)>, <Future: finished, type: numpy.ndarray, key: ('getitem-6fa12382634b1c8113b3fe2ec652ff61', 187, 0, 0)>, <Future: finished, type: numpy.ndarray, key: ('getitem-6fa12382634b1c8113b3fe2ec652ff61', 2092, 0, 0)>, <Future: finished, type: numpy.ndarray, key: ('getitem-6fa12382634b1c8113b3fe2ec652ff61', 2721, 0, 0)>, <Future: finished, type: numpy.ndarray, key: ('getitem-19933a3497a934b3e572499345f54460', 1014, 0, 0)>, <Future: finished, type: numpy.ndarray, 

In [9]:
ds_all

<xarray.Dataset> Size: 33GB
Dimensions:     (time: 131486, lat: 125, lon: 125)
Coordinates:
  * time        (time) datetime64[ns] 1MB 2009-07-01 ... 2024-06-30T13:00:00
    local_hour  (time) int32 526kB 10 11 12 13 14 15 16 ... 17 18 19 20 21 22 23
    local_day   (time) datetime64[ns] 1MB 2009-07-01 2009-07-01 ... 2024-06-30
  * lat         (lat) float64 1kB -35.97 -35.93 -35.89 ... -31.09 -31.05 -31.01
  * lon         (lon) float64 1kB 137.0 137.1 137.1 137.1 ... 141.9 141.9 142.0
    height      float64 8B 100.0
    crs         int32 4B 0
Data variables:
    ua100m      (time, lat, lon) float64 16GB dask.array<chunksize=(44, 125, 125), meta=np.ndarray>
    va100m      (time, lat, lon) float64 16GB dask.array<chunksize=(44, 125, 125), meta=np.ndarray>
Attributes: (12/60)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1H.json
    productive_version:        500df2a
    variable_version:          v20240809
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    date_modified:             2024-10-11T00:33:01Z
    date_metadata_modified:    2024-10-11T00:33:01Z
    history:                   Tue Jun 11 23:05:43 2024: /g/data/access/ngm/m...
    references:                https://doi.org/10.25914/1x6g-2v48
    license:                   https://doi.org/10.25914/1x6g-2v48
    acknowledgement:           The production of BARRA2 was supported with fu...

In [10]:
# Vectorized local-day filtering
hw_days = pd.to_datetime(hw_dates['date']).dt.normalize()
bl_days = pd.to_datetime(bl_dates['date']).dt.normalize()

day_vals = pd.to_datetime(ds_all['local_day'].values).normalize()
hw_mask = pd.Index(day_vals).isin(hw_days)
bl_mask = pd.Index(day_vals).isin(bl_days)

ds_hw = ds_all.isel(time=hw_mask)
ds_bl = ds_all.isel(time=bl_mask)

# Keep only days with all 24 hours (compute indexers to avoid dask-bool indexing)
counts_bl = ds_bl['ua100m'].groupby('local_day').count('time').compute()
good_days_bl = counts_bl.where(counts_bl == 24, drop=True).local_day.to_index()
mask_bl_full = np.isin(ds_bl['local_day'].values, good_days_bl)
ds_bl = ds_bl.isel(time=mask_bl_full)

counts_hw = ds_hw['ua100m'].groupby('local_day').count('time').compute()
good_days_hw = counts_hw.where(counts_hw == 24, drop=True).local_day.to_index()
mask_hw_full = np.isin(ds_hw['local_day'].values, good_days_hw)
ds_hw = ds_hw.isel(time=mask_hw_full)

print("BL min/max hours per day:", int(counts_bl.min()), int(counts_bl.max()))
print("HW min/max hours per day:", int(counts_hw.min()), int(counts_hw.max()))

BL min/max hours per day: 14 24
HW min/max hours per day: 24 24


In [11]:
# Reshape to add hour and date components to our time coordinate (to bootstrap by day, for each hour).
ds_bl_reshaped = (
    ds_bl
    .assign_coords(
        day=('time', ds_bl['local_day'].data),
        hour=('time', ds_bl['local_hour'].data)
    )
    .set_index(time=['day','hour'])
    .unstack('time')
)

ds_hw_reshaped = (
    ds_hw
    .assign_coords(
        day=('time', ds_hw['local_day'].data),
        hour=('time', ds_hw['local_hour'].data)
    )
    .set_index(time=['day','hour'])
    .unstack('time')
)

# Adjust chunks (tune as needed)
ds_hw_chunked = ds_hw_reshaped.chunk({'day': 160, 'hour': -1, 'lat': 25, 'lon': -1})
ds_bl_chunked = ds_bl_reshaped.chunk({'day': 160, 'hour': -1, 'lat': 25, 'lon': -1})

In [12]:
%%time
ds_hw = ds_hw_chunked.persist()
ds_bl = ds_bl_chunked.persist()

wait(ds_hw)
print('Heatwave input data ready for bootstrap')
wait(ds_bl)
print('Baseline input data ready for bootstrap')

ds_bl

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 10.64 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Heatwave input data ready for bootstrap
Baseline input data ready for bootstrap
CPU times: user 56.3 s, sys: 4.64 s, total: 1min
Wall time: 1min 28s


<xarray.Dataset> Size: 33GB
Dimensions:     (day: 5430, hour: 24, lat: 125, lon: 125)
Coordinates:
  * day         (day) datetime64[ns] 43kB 2009-07-02 2009-07-03 ... 2024-06-30
  * hour        (hour) int32 96B 0 1 2 3 4 5 6 7 8 ... 16 17 18 19 20 21 22 23
    local_hour  (day, hour) int32 521kB dask.array<chunksize=(160, 24), meta=np.ndarray>
    local_day   (day, hour) datetime64[ns] 1MB dask.array<chunksize=(160, 24), meta=np.ndarray>
  * lat         (lat) float64 1kB -35.97 -35.93 -35.89 ... -31.09 -31.05 -31.01
  * lon         (lon) float64 1kB 137.0 137.1 137.1 137.1 ... 141.9 141.9 142.0
    height      float64 8B 100.0
    crs         int32 4B 0
Data variables:
    ua100m      (lat, lon, day, hour) float64 16GB dask.array<chunksize=(25, 125, 160, 24), meta=np.ndarray>
    va100m      (lat, lon, day, hour) float64 16GB dask.array<chunksize=(25, 125, 160, 24), meta=np.ndarray>
Attributes: (12/60)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1H.json
    productive_version:        500df2a
    variable_version:          v20240809
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    date_modified:             2024-10-11T00:33:01Z
    date_metadata_modified:    2024-10-11T00:33:01Z
    history:                   Tue Jun 11 23:05:43 2024: /g/data/access/ngm/m...
    references:                https://doi.org/10.25914/1x6g-2v48
    license:                   https://doi.org/10.25914/1x6g-2v48
    acknowledgement:           The production of BARRA2 was supported with fu...

In [13]:
n_boot = 1000
sample_size = ds_hw_chunked.coords['day'].size
sample_size

50

In [14]:
%%time

def prepare_dataset(ds, day_chunk=160, lat_chunk=-1):
    # Expect dims: day, hour, lat, lon
    # Just ensure chunking & maybe convert to float32 lazily
    target = ds.chunk({'day': day_chunk, 'hour': -1, 'lat': lat_chunk, 'lon': -1})
    return target


ds_hw_p = prepare_dataset(ds_hw, day_chunk=160, lat_chunk=25).persist()
ds_bl_p = prepare_dataset(ds_bl, day_chunk=160, lat_chunk=25).persist()

CPU times: user 60.7 ms, sys: 2.4 ms, total: 63.1 ms
Wall time: 61.7 ms


In [15]:
# def bootstrap_pair_uv_summary(
#     ds_hw,
#     ds_bl,
#     vars_uv=("ua100m", "va100m"),
#     n_boot=1000,
#     batch=200,
#     sample_size=None,
#     day_dim="day",
#     seed=42,
#     ci_level=0.95,
#     direction_units="rad",
#     compute_pvalues=True,
#     check_coords=True,
#     use_float32=True,
#     debug=False,
#     debug_peek=False,  # if True, stop after first batch
# ):
#     """
#     Streaming multinomial bootstrap of (u,v) means for two groups (hw, bl) plus vector differences.

#     Parameters
#     ----------
#     ds_hw, ds_bl : xarray.Dataset
#         Must contain the u,v component variables.
#     vars_uv : (str, str)
#         Names of (u, v) component variables.
#     n_boot : int
#         Total bootstrap replicates (drawn with multinomial weights).
#     batch : int
#         Number of bootstrap replicates processed per streaming chunk.
#     sample_size : int or None
#         Bootstrap sample size (number of resampled days). If None, defaults to n_hw.
#     day_dim : str
#         Name of the day/sample dimension.
#     seed : int
#         RNG seed.
#     ci_level : float
#         Confidence level for normal CIs.
#     direction_units : {'rad','deg'}
#         Units for direction outputs.
#     compute_pvalues : bool
#         Whether to compute vector magnitude and joint component p-values.
#     check_coords : bool
#         If True, assert non-day coordinate equality between ds_hw and ds_bl for vars.
#     use_float32 : bool
#         Cast inputs to float32 before bootstrap for memory/time.
#     debug : bool
#         If True, print diagnostics for first batch.
#     debug_peek : bool
#         If True and debug, exit after first batch (useful for rapid inspection).

#     Returns
#     -------
#     xr.Dataset with component means, std, CIs, speed metrics, direction, differences, and (optionally) p-values.
#     """
#     if len(vars_uv) != 2:
#         raise ValueError("vars_uv must be length 2 (u, v).")
#     u_name, v_name = vars_uv
#     for v in vars_uv:
#         if v not in ds_hw or v not in ds_bl:
#             raise ValueError(f"Missing variable {v} in one dataset.")

#     if use_float32:
#         ds_hw = ds_hw.astype("float32", copy=False)
#         ds_bl = ds_bl.astype("float32", copy=False)

#     # Coordinate consistency checks
#     if check_coords:
#         for var in vars_uv:
#             hw_var = ds_hw[var]
#             bl_var = ds_bl[var]
#             for d in hw_var.dims:
#                 if d == day_dim:
#                     continue
#                 if d not in bl_var.dims:
#                     raise ValueError(f"Baseline missing dim '{d}' present in hw for var '{var}'.")
#                 if not hw_var.indexes[d].equals(bl_var.indexes[d]):
#                     raise ValueError(f"Coordinate mismatch on dim '{d}' for var '{var}'.")

#     n_hw = ds_hw.sizes[day_dim]
#     n_bl = ds_bl.sizes[day_dim]

#     if sample_size is None:
#         sample_size = min(n_hw,n_bl)

#     rng = np.random.default_rng(seed)

#     scopes = ("hw", "bl")
#     sum_u  = {s: None for s in scopes}
#     sum_v  = {s: None for s in scopes}
#     sum_u2 = {s: None for s in scopes}
#     sum_v2 = {s: None for s in scopes}
#     sum_uv = {s: None for s in scopes}

#     sum_diff_u = sum_diff_v = None
#     sum_diff_u2 = sum_diff_v2 = None
#     sum_diff_uv = None

#     if compute_pvalues:
#         vector_mag_ge_obs = None
#         u_diff_abs_ge_obs = None
#         v_diff_abs_ge_obs = None

#     done = 0
#     while done < n_boot:
#         b = min(batch, n_boot - done)

#         counts_hw = rng.multinomial(sample_size, np.full(n_hw, 1 / n_hw), size=b)
#         w_hw = xr.DataArray(
#             (counts_hw / sample_size).astype("float32"),
#             dims=("boot", day_dim),
#             coords={"boot": np.arange(done, done + b), day_dim: ds_hw[day_dim]},
#         )
#         counts_bl = rng.multinomial(sample_size, np.full(n_bl, 1 / n_bl), size=b)
#         w_bl = xr.DataArray(
#             (counts_bl / sample_size).astype("float32"),
#             dims=("boot", day_dim),
#             coords={"boot": np.arange(done, done + b), day_dim: ds_bl[day_dim]},
#         )

#         # Compute bootstrap weighted means
#         u_hw_batch = xr.dot(w_hw, ds_hw[u_name], dims=day_dim)
#         v_hw_batch = xr.dot(w_hw, ds_hw[v_name], dims=day_dim)
#         u_bl_batch = xr.dot(w_bl, ds_bl[u_name], dims=day_dim)
#         v_bl_batch = xr.dot(w_bl, ds_bl[v_name], dims=day_dim)

#         u_diff_batch = u_hw_batch - u_bl_batch
#         v_diff_batch = v_hw_batch - v_bl_batch
        

#         if debug and done == 0:
#             # Show a small slice
#             slice_hours = slice(0, min(6, u_bl_batch.sizes.get("hour", 6)))
#             try:
#                 print("DEBUG: baseline u first hours (boot=0):",
#                       u_bl_batch.isel(boot=0).isel(hour=slice_hours).values)
#                 print("DEBUG: hw u first hours (boot=0):",
#                       u_hw_batch.isel(boot=0).isel(hour=slice_hours).values)
#                 print("DEBUG: diff u first hours (boot=0):",
#                       u_diff_batch.isel(boot=0).isel(hour=slice_hours).values)
#             except Exception as e:
#                 print("DEBUG print failed (maybe no 'hour' dim):", repr(e))

#         # Reduce each scope
#         for scope, (u_b, v_b) in {"hw": (u_hw_batch, v_hw_batch),
#                                   "bl": (u_bl_batch, v_bl_batch)}.items():
#             su  = u_b.sum("boot").compute()
#             sv  = v_b.sum("boot").compute()
#             su2 = (u_b**2).sum("boot").compute()
#             sv2 = (v_b**2).sum("boot").compute()
#             suv = (u_b * v_b).sum("boot").compute()
#             if sum_u[scope] is None:
#                 sum_u[scope], sum_v[scope] = su, sv
#                 sum_u2[scope], sum_v2[scope] = su2, sv2
#                 sum_uv[scope] = suv
#             else:
#                 sum_u[scope]  += su
#                 sum_v[scope]  += sv
#                 sum_u2[scope] += su2
#                 sum_v2[scope] += sv2
#                 sum_uv[scope] += suv

#         # Difference accumulators
#         su_diff  = u_diff_batch.sum("boot").compute()
#         sv_diff  = v_diff_batch.sum("boot").compute()
#         su2_diff = (u_diff_batch**2).sum("boot").compute()
#         sv2_diff = (v_diff_batch**2).sum("boot").compute()
#         suv_diff = (u_diff_batch * v_diff_batch).sum("boot").compute()

#         if sum_diff_u is None:
#             sum_diff_u, sum_diff_v = su_diff, sv_diff
#             sum_diff_u2, sum_diff_v2 = su2_diff, sv2_diff
#             sum_diff_uv = suv_diff
#         else:
#             sum_diff_u  += su_diff
#             sum_diff_v  += sv_diff
#             sum_diff_u2 += su2_diff
#             sum_diff_v2 += sv2_diff
#             sum_diff_uv += suv_diff

#         # P-values
#         if compute_pvalues:
#             u_mean_running = sum_diff_u / (done + b)
#             v_mean_running = sum_diff_v / (done + b)
#             vector_mag_batch = (u_diff_batch**2 + v_diff_batch**2)**0.5
#             vector_mag_obs = (u_mean_running**2 + v_mean_running**2)**0.5
#             vector_ge = (vector_mag_batch >= vector_mag_obs).astype("float32").sum("boot").compute()
#             u_abs_ge = (xr.ufuncs.fabs(u_diff_batch) >= xr.ufuncs.fabs(u_mean_running)).astype("float32").sum("boot").compute()
#             v_abs_ge = (xr.ufuncs.fabs(v_diff_batch) >= xr.ufuncs.fabs(v_mean_running)).astype("float32").sum("boot").compute()
#             if vector_mag_ge_obs is None:
#                 vector_mag_ge_obs = vector_ge
#                 u_diff_abs_ge_obs = u_abs_ge
#                 v_diff_abs_ge_obs = v_abs_ge
#             else:
#                 vector_mag_ge_obs += vector_ge
#                 u_diff_abs_ge_obs += u_abs_ge
#                 v_diff_abs_ge_obs += v_abs_ge

#         done += b
#         if debug:
#             print(f"DEBUG: completed batch; cumulative {done}/{n_boot}")
#         if debug and debug_peek:
#             print("DEBUG: peek mode stopping after first batch.")
#             break

#     # z for CI
#     # Support general ci_level (normal approx)
#     from math import erf, sqrt
#     # If ci_level is standard, pick constants; else use inverse error function
#     if abs(ci_level - 0.95) < 1e-12:
#         z = 1.959963984540054
#     elif abs(ci_level - 0.90) < 1e-12:
#         z = 1.6448536269514722
#     elif abs(ci_level - 0.99) < 1e-12:
#         z = 2.5758293035489004
#     else:
#         # approximate inverse CDF via binary search or use scipy if available
#         try:
#             from scipy.stats import norm
#             z = norm.ppf(0.5 + ci_level / 2.0)
#         except Exception:
#             # crude fallback (not critical if you only use standard levels)
#             z = 1.959963984540054

#     N = done if (debug and debug_peek) else n_boot
#     out = {}

#     def finalize(scope, prefix):
#         u_sum = sum_u[scope]
#         v_sum = sum_v[scope]
#         u2_sum = sum_u2[scope]
#         v2_sum = sum_v2[scope]
#         uv_sum = sum_uv[scope]

#         u_mean = u_sum / N
#         v_mean = v_sum / N
#         u_var = (u2_sum - (u_sum**2) / N) / (N - 1)
#         v_var = (v2_sum - (v_sum**2) / N) / (N - 1)
#         uv_cov = (uv_sum - (u_sum * v_sum) / N) / (N - 1)

#         u_var = xr.where(u_var < 0, 0, u_var)
#         v_var = xr.where(v_var < 0, 0, v_var)

#         u_std = u_var**0.5
#         v_std = v_var**0.5

#         out[f"{prefix}_u_mean"] = u_mean
#         out[f"{prefix}_u_std"] = u_std
#         out[f"{prefix}_u_ci_low"] = u_mean - z * u_std
#         out[f"{prefix}_u_ci_high"] = u_mean + z * u_std

#         out[f"{prefix}_v_mean"] = v_mean
#         out[f"{prefix}_v_std"] = v_std
#         out[f"{prefix}_v_ci_low"] = v_mean - z * v_std
#         out[f"{prefix}_v_ci_high"] = v_mean + z * v_std

#         speed_mean = (u_mean**2 + v_mean**2)**0.5
#         eps = 1e-20
#         denom = xr.where(speed_mean == 0, eps, speed_mean)
#         speed_var = (
#             (u_mean / denom)**2 * u_var
#             + (v_mean / denom)**2 * v_var
#             + 2 * (u_mean * v_mean) / (denom**2) * uv_cov
#         )
#         speed_var = xr.where(speed_var < 0, 0, speed_var)
#         speed_std = speed_var**0.5

#         out[f"{prefix}_speed_mean"] = speed_mean
#         out[f"{prefix}_speed_std"] = speed_std
#         out[f"{prefix}_speed_ci_low"] = speed_mean - z * speed_std
#         out[f"{prefix}_speed_ci_high"] = speed_mean + z * speed_std

#         direction_mean = xr.apply_ufunc(np.arctan2, v_mean, u_mean)
#         s2 = denom**2
#         s4 = s2**2
#         dir_var = (
#             (v_mean**2) * u_var
#             + (u_mean**2) * v_var
#             - 2 * u_mean * v_mean * uv_cov
#         ) / xr.where(s4 == 0, eps, s4)
#         dir_var = xr.where(dir_var < 0, 0, dir_var)
#         dir_std = dir_var**0.5
#         dir_ci_low = direction_mean - z * dir_std
#         dir_ci_high = direction_mean + z * dir_std

#         if direction_units == "deg":
#             factor = 180.0 / np.pi
#             direction_mean = direction_mean * factor
#             dir_std = dir_std * factor
#             dir_ci_low = dir_ci_low * factor
#             dir_ci_high = dir_ci_high * factor

#         out[f"{prefix}_direction_mean"] = direction_mean
#         out[f"{prefix}_direction_std"] = dir_std
#         out[f"{prefix}_direction_ci_low"] = dir_ci_low
#         out[f"{prefix}_direction_ci_high"] = dir_ci_high
#         out[f"{prefix}_uv_cov"] = uv_cov

#         return u_mean, v_mean, speed_mean

#     u_hw_mean, v_hw_mean, speed_hw_mean = finalize("hw", "hw")
#     u_bl_mean, v_bl_mean, speed_bl_mean = finalize("bl", "bl")

#     # Differences
#     u_diff_mean = sum_diff_u / N
#     v_diff_mean = sum_diff_v / N
#     u_diff_var = (sum_diff_u2 - (sum_diff_u**2)/N) / (N - 1)
#     v_diff_var = (sum_diff_v2 - (sum_diff_v**2)/N) / (N - 1)
#     uv_diff_cov = (sum_diff_uv - (sum_diff_u * sum_diff_v)/N) / (N - 1)

#     u_diff_var = xr.where(u_diff_var < 0, 0, u_diff_var)
#     v_diff_var = xr.where(v_diff_var < 0, 0, v_diff_var)

#     speed_diff_vec_mean = (u_diff_mean**2 + v_diff_mean**2)**0.5
#     eps = 1e-20
#     denom_diff = xr.where(speed_diff_vec_mean == 0, eps, speed_diff_vec_mean)
#     speed_diff_var = (
#         (u_diff_mean / denom_diff)**2 * u_diff_var
#         + (v_diff_mean / denom_diff)**2 * v_diff_var
#         + 2 * (u_diff_mean * v_diff_mean)/(denom_diff**2) * uv_diff_cov
#     )
#     speed_diff_var = xr.where(speed_diff_var < 0, 0, speed_diff_var)
#     speed_diff_std = speed_diff_var**0.5

#     direction_diff_mean = xr.apply_ufunc(np.arctan2, v_diff_mean, u_diff_mean)
#     s2_diff = denom_diff**2
#     s4_diff = s2_diff**2
#     dir_diff_var = (
#         (v_diff_mean**2) * u_diff_var
#         + (u_diff_mean**2) * v_diff_var
#         - 2 * u_diff_mean * v_diff_mean * uv_diff_cov
#     ) / xr.where(s4_diff == 0, eps, s4_diff)
#     dir_diff_var = xr.where(dir_diff_var < 0, 0, dir_diff_var)
#     dir_diff_std = dir_diff_var**0.5
#     dir_ci_low_diff = direction_diff_mean - z * dir_diff_std
#     dir_ci_high_diff = direction_diff_mean + z * dir_diff_std

#     if direction_units == "deg":
#         factor = 180.0 / np.pi
#         direction_diff_mean = direction_diff_mean * factor
#         dir_diff_std = dir_diff_std * factor
#         dir_ci_low_diff = dir_ci_low_diff * factor
#         dir_ci_high_diff = dir_ci_high_diff * factor

#     out["diff_u_mean"] = u_diff_mean
#     out["diff_v_mean"] = v_diff_mean
#     out["diff_speed_mean"] = speed_diff_vec_mean
#     out["diff_speed_std"] = speed_diff_std
#     out["diff_speed_ci_low"] = speed_diff_vec_mean - z * speed_diff_std
#     out["diff_speed_ci_high"] = speed_diff_vec_mean + z * speed_diff_std
#     out["diff_direction_mean"] = direction_diff_mean
#     out["diff_direction_std"] = dir_diff_std
#     out["diff_direction_ci_low"] = dir_ci_low_diff
#     out["diff_direction_ci_high"] = dir_ci_high_diff
#     out["diff_uv_cov"] = uv_diff_cov

#     if compute_pvalues:
#         vector_pval = vector_mag_ge_obs / N
#         u_pval = u_diff_abs_ge_obs / N
#         v_pval = v_diff_abs_ge_obs / N
#         joint_pval = xr.ufuncs.maximum(u_pval, v_pval)
#         out["diff_magnitude_pval"] = vector_pval
#         out["diff_joint_pval"] = joint_pval
#         out["diff_sig_magnitude_005"] = vector_pval < 0.05
#         out["diff_sig_magnitude_001"] = vector_pval < 0.01
#         out["diff_sig_joint_005"] = joint_pval < 0.05
#         out["diff_sig_joint_001"] = joint_pval < 0.01

#     return xr.Dataset(out)

In [16]:
def bootstrap_pair_uv_summary(
    ds_hw,
    ds_bl,
    vars_uv=("ua100m", "va100m"),
    n_boot=1000,
    batch=200,
    sample_size=None,
    day_dim="day",
    seed=42,
    ci_level=0.95,
    direction_units="rad",
    compute_pvalues=True,
    check_coords=True,
    use_float32=True,
    debug=False,
    debug_peek=False,
):
    """
    Streaming multinomial bootstrap of (u,v) means for two groups (hw, bl) plus vector differences.

    Parameters
    ----------
    ds_hw, ds_bl : xarray.Dataset
        Must contain the u,v component variables.
    vars_uv : (str, str)
        Names of (u, v) component variables.
    n_boot : int
        Total bootstrap replicates (drawn with multinomial weights).
    batch : int
        Number of bootstrap replicates processed per streaming chunk.
    sample_size : int or None
        Bootstrap sample size (number of resampled days). If None, defaults to n_hw.
    day_dim : str
        Name of the day/sample dimension.
    seed : int
        RNG seed.
    ci_level : float
        Confidence level for normal CIs.
    direction_units : {'rad','deg'}
        Units for direction outputs.
    compute_pvalues : bool
        Whether to compute vector magnitude and joint component p-values.
    check_coords : bool
        If True, assert non-day coordinate equality between ds_hw and ds_bl for vars.
    use_float32 : bool
        Cast inputs to float32 before bootstrap for memory/time.
    debug : bool
        If True, print diagnostics for first batch.
    debug_peek : bool
        If True and debug, exit after first batch (useful for rapid inspection).

    Returns
    -------
    xr.Dataset with component means, std, CIs, speed metrics, direction, differences, and (optionally) p-values.
    """
    if len(vars_uv) != 2:
        raise ValueError("vars_uv must be length 2 (u, v).")
    u_name, v_name = vars_uv
    for v in vars_uv:
        if v not in ds_hw or v not in ds_bl:
            raise ValueError(f"Missing variable {v} in one dataset.")

    if use_float32:
        ds_hw = ds_hw.astype("float32", copy=False)
        ds_bl = ds_bl.astype("float32", copy=False)

    # Coordinate consistency checks
    if check_coords:
        for var in vars_uv:
            hw_var = ds_hw[var]
            bl_var = ds_bl[var]
            for d in hw_var.dims:
                if d == day_dim:
                    continue
                if d not in bl_var.dims:
                    raise ValueError(f"Baseline missing dim '{d}' present in hw for var '{var}'.")
                if not hw_var.indexes[d].equals(bl_var.indexes[d]):
                    raise ValueError(f"Coordinate mismatch on dim '{d}' for var '{var}'.")

    n_hw = ds_hw.sizes[day_dim]
    n_bl = ds_bl.sizes[day_dim]

    if sample_size is None:
        sample_size = min(n_hw, n_bl)

    rng = np.random.default_rng(seed)

    scopes = ("hw", "bl")
    sum_u  = {s: None for s in scopes}
    sum_v  = {s: None for s in scopes}
    sum_u2 = {s: None for s in scopes}
    sum_v2 = {s: None for s in scopes}
    sum_uv = {s: None for s in scopes}

    sum_diff_u = sum_diff_v = None
    sum_diff_u2 = sum_diff_v2 = None
    sum_diff_uv = None

    if compute_pvalues:
        vector_mag_ge_obs = None
        u_diff_abs_ge_obs = None
        v_diff_abs_ge_obs = None

    sum_speed_diff = None

    done = 0
    while done < n_boot:
        b = min(batch, n_boot - done)

        counts_hw = rng.multinomial(sample_size, np.full(n_hw, 1 / n_hw), size=b)
        w_hw = xr.DataArray(
            (counts_hw / sample_size).astype("float32"),
            dims=("boot", day_dim),
            coords={"boot": np.arange(done, done + b), day_dim: ds_hw[day_dim]},
        )
        counts_bl = rng.multinomial(sample_size, np.full(n_bl, 1 / n_bl), size=b)
        w_bl = xr.DataArray(
            (counts_bl / sample_size).astype("float32"),
            dims=("boot", day_dim),
            coords={"boot": np.arange(done, done + b), day_dim: ds_bl[day_dim]},
        )

        # Compute bootstrap weighted means
        u_hw_batch = xr.dot(w_hw, ds_hw[u_name], dims=day_dim)
        v_hw_batch = xr.dot(w_hw, ds_hw[v_name], dims=day_dim)
        u_bl_batch = xr.dot(w_bl, ds_bl[u_name], dims=day_dim)
        v_bl_batch = xr.dot(w_bl, ds_bl[v_name], dims=day_dim)

        u_diff_batch = u_hw_batch - u_bl_batch
        v_diff_batch = v_hw_batch - v_bl_batch

        speed_hw_batch = (u_hw_batch**2 + v_hw_batch**2)**0.5  # ADDED
        speed_bl_batch = (u_bl_batch**2 + v_bl_batch**2)**0.5  # ADDED
        speed_diff_batch = speed_hw_batch - speed_bl_batch      # ADDED


        if sum_speed_diff is None:
            sum_speed_diff = speed_diff_batch.sum("boot").compute()  # ADDED
        else:
            sum_speed_diff += speed_diff_batch.sum("boot").compute()  # ADDED

        if debug and done == 0:
            # Show a small slice
            slice_hours = slice(0, min(6, u_bl_batch.sizes.get("hour", 6)))
            try:
                print("DEBUG: baseline u first hours (boot=0):",
                      u_bl_batch.isel(boot=0).isel(hour=slice_hours).values)
                print("DEBUG: hw u first hours (boot=0):",
                      u_hw_batch.isel(boot=0).isel(hour=slice_hours).values)
                print("DEBUG: diff u first hours (boot=0):",
                      u_diff_batch.isel(boot=0).isel(hour=slice_hours).values)
            except Exception as e:
                print("DEBUG print failed (maybe no 'hour' dim):", repr(e))

        # Reduce each scope
        for scope, (u_b, v_b) in {"hw": (u_hw_batch, v_hw_batch),
                                  "bl": (u_bl_batch, v_bl_batch)}.items():
            su  = u_b.sum("boot").compute()
            sv  = v_b.sum("boot").compute()
            su2 = (u_b**2).sum("boot").compute()
            sv2 = (v_b**2).sum("boot").compute()
            suv = (u_b * v_b).sum("boot").compute()
            if sum_u[scope] is None:
                sum_u[scope], sum_v[scope] = su, sv
                sum_u2[scope], sum_v2[scope] = su2, sv2
                sum_uv[scope] = suv
            else:
                sum_u[scope]  += su
                sum_v[scope]  += sv
                sum_u2[scope] += su2
                sum_v2[scope] += sv2
                sum_uv[scope] += suv

        # Difference accumulators
        su_diff  = u_diff_batch.sum("boot").compute()
        sv_diff  = v_diff_batch.sum("boot").compute()
        su2_diff = (u_diff_batch**2).sum("boot").compute()
        sv2_diff = (v_diff_batch**2).sum("boot").compute()
        suv_diff = (u_diff_batch * v_diff_batch).sum("boot").compute()

        if sum_diff_u is None:
            sum_diff_u, sum_diff_v = su_diff, sv_diff
            sum_diff_u2, sum_diff_v2 = su2_diff, sv2_diff
            sum_diff_uv = suv_diff
        else:
            sum_diff_u  += su_diff
            sum_diff_v  += sv_diff
            sum_diff_u2 += su2_diff
            sum_diff_v2 += sv2_diff
            sum_diff_uv += suv_diff

        # P-values
        if compute_pvalues:
            u_mean_running = sum_diff_u / (done + b)
            v_mean_running = sum_diff_v / (done + b)
            vector_mag_batch = (u_diff_batch**2 + v_diff_batch**2)**0.5
            vector_mag_obs = (u_mean_running**2 + v_mean_running**2)**0.5
            vector_ge = (vector_mag_batch >= vector_mag_obs).astype("float32").sum("boot").compute()
            u_abs_ge = (xr.ufuncs.fabs(u_diff_batch) >= xr.ufuncs.fabs(u_mean_running)).astype("float32").sum("boot").compute()
            v_abs_ge = (xr.ufuncs.fabs(v_diff_batch) >= xr.ufuncs.fabs(v_mean_running)).astype("float32").sum("boot").compute()
            if vector_mag_ge_obs is None:
                vector_mag_ge_obs = vector_ge
                u_diff_abs_ge_obs = u_abs_ge
                v_diff_abs_ge_obs = v_abs_ge
            else:
                vector_mag_ge_obs += vector_ge
                u_diff_abs_ge_obs += u_abs_ge
                v_diff_abs_ge_obs += v_abs_ge

        done += b
        if debug:
            print(f"DEBUG: completed batch; cumulative {done}/{n_boot}")
        if debug and debug_peek:
            print("DEBUG: peek mode stopping after first batch.")
            break

    # z for CI
    # Support general ci_level (normal approx)
    from math import erf, sqrt
    # If ci_level is standard, pick constants; else use inverse error function
    if abs(ci_level - 0.95) < 1e-12:
        z = 1.959963984540054
    elif abs(ci_level - 0.90) < 1e-12:
        z = 1.6448536269514722
    elif abs(ci_level - 0.99) < 1e-12:
        z = 2.5758293035489004
    else:
        try:
            from scipy.stats import norm
            z = norm.ppf(0.5 + ci_level / 2.0)
        except Exception:
            z = 1.959963984540054

    N = done if (debug and debug_peek) else n_boot
    out = {}

    def finalize(scope, prefix):
        u_sum = sum_u[scope]
        v_sum = sum_v[scope]
        u2_sum = sum_u2[scope]
        v2_sum = sum_v2[scope]
        uv_sum = sum_uv[scope]

        u_mean = u_sum / N
        v_mean = v_sum / N
        u_var = (u2_sum - (u_sum**2) / N) / (N - 1)
        v_var = (v2_sum - (v_sum**2) / N) / (N - 1)
        uv_cov = (uv_sum - (u_sum * v_sum) / N) / (N - 1)

        u_var = xr.where(u_var < 0, 0, u_var)
        v_var = xr.where(v_var < 0, 0, v_var)

        u_std = u_var**0.5
        v_std = v_var**0.5

        out[f"{prefix}_u_mean"] = u_mean
        out[f"{prefix}_u_std"] = u_std
        out[f"{prefix}_u_ci_low"] = u_mean - z * u_std
        out[f"{prefix}_u_ci_high"] = u_mean + z * u_std

        out[f"{prefix}_v_mean"] = v_mean
        out[f"{prefix}_v_std"] = v_std
        out[f"{prefix}_v_ci_low"] = v_mean - z * v_std
        out[f"{prefix}_v_ci_high"] = v_mean + z * v_std

        speed_mean = (u_mean**2 + v_mean**2)**0.5
        eps = 1e-20
        denom = xr.where(speed_mean == 0, eps, speed_mean)
        speed_var = (
            (u_mean / denom)**2 * u_var
            + (v_mean / denom)**2 * v_var
            + 2 * (u_mean * v_mean) / (denom**2) * uv_cov
        )
        speed_var = xr.where(speed_var < 0, 0, speed_var)
        speed_std = speed_var**0.5

        out[f"{prefix}_speed_mean"] = speed_mean
        out[f"{prefix}_speed_std"] = speed_std
        out[f"{prefix}_speed_ci_low"] = speed_mean - z * speed_std
        out[f"{prefix}_speed_ci_high"] = speed_mean + z * speed_std

        direction_mean = xr.apply_ufunc(np.arctan2, v_mean, u_mean)
        s2 = denom**2
        s4 = s2**2
        dir_var = (
            (v_mean**2) * u_var
            + (u_mean**2) * v_var
            - 2 * u_mean * v_mean * uv_cov
        ) / xr.where(s4 == 0, eps, s4)
        dir_var = xr.where(dir_var < 0, 0, dir_var)
        dir_std = dir_var**0.5
        dir_ci_low = direction_mean - z * dir_std
        dir_ci_high = direction_mean + z * dir_std

        if direction_units == "deg":
            factor = 180.0 / np.pi
            direction_mean = direction_mean * factor
            dir_std = dir_std * factor
            dir_ci_low = dir_ci_low * factor
            dir_ci_high = dir_ci_high * factor

        out[f"{prefix}_direction_mean"] = direction_mean
        out[f"{prefix}_direction_std"] = dir_std
        out[f"{prefix}_direction_ci_low"] = dir_ci_low
        out[f"{prefix}_direction_ci_high"] = dir_ci_high
        out[f"{prefix}_uv_cov"] = uv_cov

        return u_mean, v_mean, speed_mean

    u_hw_mean, v_hw_mean, speed_hw_mean = finalize("hw", "hw")
    u_bl_mean, v_bl_mean, speed_bl_mean = finalize("bl", "bl")

    # Differences
    u_diff_mean = sum_diff_u / N
    v_diff_mean = sum_diff_v / N
    u_diff_var = (sum_diff_u2 - (sum_diff_u**2)/N) / (N - 1)
    v_diff_var = (sum_diff_v2 - (sum_diff_v**2)/N) / (N - 1)
    uv_diff_cov = (sum_diff_uv - (sum_diff_u * sum_diff_v)/N) / (N - 1)

    u_diff_var = xr.where(u_diff_var < 0, 0, u_diff_var)
    v_diff_var = xr.where(v_diff_var < 0, 0, v_diff_var)

    speed_diff_vec_mean = (u_diff_mean**2 + v_diff_mean**2)**0.5
    eps = 1e-20
    denom_diff = xr.where(speed_diff_vec_mean == 0, eps, speed_diff_vec_mean)
    speed_diff_var = (
        (u_diff_mean / denom_diff)**2 * u_diff_var
        + (v_diff_mean / denom_diff)**2 * v_diff_var
        + 2 * (u_diff_mean * v_diff_mean)/(denom_diff**2) * uv_diff_cov
    )
    speed_diff_var = xr.where(speed_diff_var < 0, 0, speed_diff_var)
    speed_diff_std = speed_diff_var**0.5

    direction_diff_mean = xr.apply_ufunc(np.arctan2, v_diff_mean, u_diff_mean)
    s2_diff = denom_diff**2
    s4_diff = s2_diff**2
    dir_diff_var = (
        (v_diff_mean**2) * u_diff_var
        + (u_diff_mean**2) * v_diff_var
        - 2 * u_diff_mean * v_diff_mean * uv_diff_cov
    ) / xr.where(s4_diff == 0, eps, s4_diff)
    dir_diff_var = xr.where(dir_diff_var < 0, 0, dir_diff_var)
    dir_diff_std = dir_diff_var**0.5
    dir_ci_low_diff = direction_diff_mean - z * dir_diff_std
    dir_ci_high_diff = direction_diff_mean + z * dir_diff_std

    if direction_units == "deg":
        factor = 180.0 / np.pi
        direction_diff_mean = direction_diff_mean * factor
        dir_diff_std = dir_diff_std * factor
        dir_ci_low_diff = dir_ci_low_diff * factor
        dir_ci_high_diff = dir_ci_high_diff * factor

    out["diff_u_mean"] = u_diff_mean
    out["diff_v_mean"] = v_diff_mean
    out["diff_speed_mean"] = speed_diff_vec_mean
    out["diff_speed_std"] = speed_diff_std
    out["diff_speed_ci_low"] = speed_diff_vec_mean - z * speed_diff_std
    out["diff_speed_ci_high"] = speed_diff_vec_mean + z * speed_diff_std
    out["diff_direction_mean"] = direction_diff_mean
    out["diff_direction_std"] = dir_diff_std
    out["diff_direction_ci_low"] = dir_ci_low_diff
    out["diff_direction_ci_high"] = dir_ci_high_diff
    out["diff_uv_cov"] = uv_diff_cov

    speed_diff_mean = sum_speed_diff / N
    out["diff_speed_signed"] = speed_diff_mean

    if compute_pvalues:
        vector_pval = vector_mag_ge_obs / N
        u_pval = u_diff_abs_ge_obs / N
        v_pval = v_diff_abs_ge_obs / N
        joint_pval = xr.ufuncs.maximum(u_pval, v_pval)
        out["diff_magnitude_pval"] = vector_pval
        out["diff_joint_pval"] = joint_pval
        out["diff_sig_magnitude_005"] = vector_pval < 0.05
        out["diff_sig_magnitude_001"] = vector_pval < 0.01
        out["diff_sig_joint_005"] = joint_pval < 0.05
        out["diff_sig_joint_001"] = joint_pval < 0.01
        out["diff_u_pval"] = u_pval
        out["diff_v_pval"] = v_pval

    return xr.Dataset(out)

In [17]:
%%time

summary = bootstrap_pair_uv_summary(
    ds_hw_p,
    ds_bl_p,
    vars_uv=("ua100m","va100m"),
    n_boot=1000,
    sample_size=sample_size,
    batch=250,
    ci_level=0.95
    ).compute()

CPU times: user 2min 4s, sys: 29.3 s, total: 2min 34s
Wall time: 5min 49s


In [18]:
summary

<xarray.Dataset> Size: 77MB
Dimensions:                 (hour: 24, lon: 125, lat: 125)
Coordinates:
  * hour                    (hour) int32 96B 0 1 2 3 4 5 6 ... 18 19 20 21 22 23
  * lon                     (lon) float64 1kB 137.0 137.1 137.1 ... 141.9 142.0
  * lat                     (lat) float64 1kB -35.97 -35.93 ... -31.05 -31.01
    height                  float64 8B 100.0
    crs                     int32 4B 0
Data variables: (12/54)
    hw_u_mean               (lat, lon, hour) float32 2MB -4.447 -4.28 ... -5.075
    hw_u_std                (lat, lon, hour) float32 2MB 0.6853 ... 0.7937
    hw_u_ci_low             (lat, lon, hour) float32 2MB -5.79 -5.617 ... -6.631
    hw_u_ci_high            (lat, lon, hour) float32 2MB -3.104 -2.942 ... -3.52
    hw_v_mean               (lat, lon, hour) float32 2MB 0.1727 ... -1.309
    hw_v_std                (lat, lon, hour) float32 2MB 0.7347 0.8078 ... 1.034
    ...                      ...
    diff_sig_magnitude_005  (lat, lon, hour) bool 375kB False False ... False
    diff_sig_magnitude_001  (lat, lon, hour) bool 375kB False False ... False
    diff_sig_joint_005      (lat, lon, hour) bool 375kB False False ... False
    diff_sig_joint_001      (lat, lon, hour) bool 375kB False False ... False
    diff_u_pval             (lat, lon, hour) float32 2MB 0.492 0.511 ... 0.499
    diff_v_pval             (lat, lon, hour) float32 2MB 0.742 0.652 ... 0.504

In [19]:
%%time
encoding = {
    var: {"zlib": True, "complevel": 4, "shuffle": True}
    for var in summary.data_vars
    }

# Save heatwave bootstrap results
file = summary.to_netcdf("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/bootstrap_summary.nc",
                         encoding=encoding,
                         compute=False).compute()

CPU times: user 3.14 s, sys: 288 ms, total: 3.43 s
Wall time: 3.22 s


In [20]:
# import numpy as np
# import matplotlib.pyplot as plt

# def plot_diff_uv_map(
#     summary,
#     value_field="speed_difference_mean",  # e.g. "speed_difference_mean", "diff_u_mean", "diff_v_mean", or "magnitude"
#     hour=None,
#     avg_over_hour=False,
#     use_hotelling_if_available=True,
#     use_fdr=False,
#     strict_vector_significance=False,
#     quiver_subsample=4,
#     arrow_color="black",
#     arrow_alpha=0.85,
#     arrow_scale=100,
#     cmap_seq="viridis",
#     cmap_div="RdBu_r",
#     center_zero=True,
#     figsize=(11,6),
#     projection=None,
#     central_longitude=0,
#     coast=True,
#     title=None,
#     show_dots=True,
#     show_hatch=False,
#     stipple_color="black",
#     stipple_edge="black",
#     stipple_size=1,
#     hatch_color="none",
#     hatch_alpha=0.1,
#     hatch_pattern="///",
# ):
#     """
#     Plot chosen difference field with significance stippling/hatching.

#     value_field options:
#       - "speed_difference_mean" (signed: |V_hw| - |V_bl|)
#       - "diff_u_mean" (signed u-component diff)
#       - "diff_v_mean" (signed v-component diff)
#       - "magnitude" (|Δ vector|, always >=0; uses sequential cmap and ignores center_zero)
#       - any other diff_*_mean present in summary

#     center_zero applies only to signed fields (ignored if value_field == "magnitude").
#     """
#     try:
#         import cartopy.crs as ccrs
#         have_cartopy = True
#     except ImportError:
#         have_cartopy = False

#     # Core mean components
#     u = summary["diff_u_mean"]
#     v = summary["diff_v_mean"]

#     # CIs for component significance fallback
#     u_low  = summary["diff_u_ci_low"]
#     u_high = summary["diff_u_ci_high"]
#     v_low  = summary["diff_v_ci_low"]
#     v_high = summary["diff_v_ci_high"]

#     # Hour handling
#     def sel_or_mean(da):
#         if "hour" not in da.dims:
#             return da
#         if avg_over_hour:
#             return da.mean("hour")
#         if hour is None:
#             return da.isel(hour=0)
#         return da.sel(hour=hour)

#     u = sel_or_mean(u); v = sel_or_mean(v)
#     u_low  = sel_or_mean(u_low);  u_high = sel_or_mean(u_high)
#     v_low  = sel_or_mean(v_low);  v_high = sel_or_mean(v_high)

#     # Magnitude (for optional plotting)
#     mag = np.hypot(u, v)

#     # Decide data_field
#     if value_field == "magnitude":
#         data_field = mag
#         signed = False
#     else:
#         if value_field not in summary:
#             raise ValueError(f"{value_field} not found in summary.")
#         data_field_raw = summary[value_field]
#         data_field = sel_or_mean(data_field_raw)
#         signed = True

#     # Significance mask selection
#     sig_hot = summary.get("diff_sig_mask", None)
#     sig_fdr = summary.get("diff_fdr_mask", None)
#     if sig_hot is not None:
#         sig_hot = sel_or_mean(sig_hot)
#     if sig_fdr is not None:
#         sig_fdr = sel_or_mean(sig_fdr)

#     if use_fdr and sig_fdr is not None:
#         sig_mask = sig_fdr
#     elif use_hotelling_if_available and sig_hot is not None:
#         sig_mask = sig_hot
#     else:
#         sig_u = (u_low > 0) | (u_high < 0)
#         sig_v = (v_low > 0) | (v_high < 0)
#         sig_mask = (sig_u & sig_v) if strict_vector_significance else (sig_u | sig_v)

#     lat = u.coords["lat"]; lon = u.coords["lon"]

#     # Colormap / norm
#     import matplotlib.colors as mcolors
#     if signed and center_zero:
#         dmin = float(data_field.min())
#         dmax = float(data_field.max())
#         span = max(abs(dmin), abs(dmax)) or 1.0
#         norm = mcolors.TwoSlopeNorm(vcenter=0.0, vmin=-span, vmax=span)
#         cmap_use = cmap_div
#     else:
#         norm = None
#         cmap_use = cmap_seq

#     # Build map
#     if have_cartopy:
#         if projection is None:
#             projection = ccrs.PlateCarree(central_longitude=central_longitude)
#         data_crs = ccrs.PlateCarree()
#         fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": projection})
#         mesh = ax.pcolormesh(lon, lat, data_field, transform=data_crs, cmap=cmap_use, norm=norm)
#     else:
#         fig, ax = plt.subplots(figsize=figsize)
#         mesh = ax.pcolormesh(lon, lat, data_field, shading="auto", cmap=cmap_use, norm=norm)

#     cb_label = {
#         "magnitude": "|Δ vector| (m/s)",
#         "speed_difference_mean": "Δ speed (m/s)"
#     }.get(value_field, value_field.replace("diff_", "Δ ").replace("_mean", "").replace("_", " "))
#     plt.colorbar(mesh, ax=ax, pad=0.02, label=cb_label)

#     # Quiver (show raw vector difference)
#     sl_lat = slice(0, None, quiver_subsample)
#     sl_lon = slice(0, None, quiver_subsample)
#     if have_cartopy:
#         ax.quiver(
#             lon[sl_lon], lat[sl_lat],
#             u.isel(lon=sl_lon, lat=sl_lat),
#             v.isel(lon=sl_lon, lat=sl_lat),
#             transform=data_crs,
#             color=arrow_color, alpha=arrow_alpha,
#             scale=arrow_scale, width=0.0022, zorder=3
#         )
#     else:
#         lon2d, lat2d = np.meshgrid(lon, lat)
#         ax.quiver(
#             lon2d[sl_lat, sl_lon], lat2d[sl_lat, sl_lon],
#             u.values[sl_lat, sl_lon], v.values[sl_lat, sl_lon],
#             color=arrow_color, alpha=arrow_alpha,
#             scale=arrow_scale, width=0.0022, zorder=3
#         )

#     # Significance styling
#     if show_hatch:
#         import numpy.ma as ma
#         mask = ~sig_mask
#         hatch_data = ma.masked_where(mask, sig_mask.astype(int))
#         if have_cartopy:
#             ax.contourf(
#                 lon, lat, hatch_data, levels=[0.5,1.5],
#                 hatches=[hatch_pattern], colors=["none"],
#                 alpha=hatch_alpha,
#                 transform=data_crs,
#                 zorder=4
#             )
#         else:
#             ax.contourf(
#                 lon, lat, hatch_data, levels=[0.5,1.5],
#                 hatches=[hatch_pattern], colors=["none"],
#                 alpha=hatch_alpha,
#                 zorder=4
#             )

#     if show_dots:
#         yy, xx = np.where(sig_mask.values)
#         if have_cartopy:
#             ax.scatter(
#                 lon.values[xx], lat.values[yy],
#                 s=stipple_size, facecolors=stipple_color,
#                 edgecolors=stipple_edge, linewidths=0.3,
#                 marker="o", alpha=0.9,
#                 transform=data_crs, zorder=5
#             )
#         else:
#             ax.scatter(
#                 lon.values[xx], lat.values[yy],
#                 s=stipple_size, facecolors=stipple_color,
#                 edgecolors=stipple_edge, linewidths=0.3,
#                 marker="o", alpha=0.9, zorder=5
#             )

#     if have_cartopy and coast:
#         ax.coastlines(linewidth=0.8, zorder=6)

#     # Title
#     if title is None:
#         base = {
#             "magnitude": "|Δ wind vector|",
#             "speed_difference_mean": "Δ speed (hw - bl)"
#         }.get(value_field, cb_label)
#         title = base
#     if "hour" in summary.dims:
#         if avg_over_hour:
#             title += " (hour mean)"
#         elif hour is not None:
#             title += f" (hour={hour})"
#     ax.set_title(title)

#     # Legend
#     from matplotlib.lines import Line2D
#     legend_items = [
#         Line2D([0],[0], color=arrow_color, lw=1.5, label="Vector Δ")
#     ]
#     if show_dots:
#         legend_items.append(
#             Line2D([0],[0], marker="o", linestyle="none",
#                    markerfacecolor=stipple_color, markeredgecolor=stipple_edge,
#                    markersize=np.sqrt(stipple_size), label="Significant")
#         )
#     if show_hatch:
#         legend_items.append(
#             Line2D([0],[0], color="k", lw=0, label="Significant (hatch)")
#         )
#     ax.legend(handles=legend_items, loc="upper right", frameon=True)

#     if not have_cartopy:
#         ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

#     plt.tight_layout()
#     return fig, ax

In [21]:
# fig, ax = plot_diff_uv_map(summary,
#                            hour=12,
#                            avg_over_hour=False,
#                            value_field="speed_difference_mean",
#                            center_zero=True,
#                            cmap_div="RdBu_r",
#                            show_dots=True,
#                            show_hatch=False)

In [22]:
# import numpy as np
# import xarray as xr
# import matplotlib.pyplot as plt
# from scipy.stats import chi2

# def plot_mean_wind(summary,
#                    scope="hw",              # "hw" or "bl"
#                    hour=None,               # integer hour to select if hour dim exists
#                    avg_over_hour=False,     # if True, ignore 'hour' and average
#                    show_significance=True,
#                    sig_alpha=0.05,
#                    strict_vector=False,     # if covariance singular: require BOTH u & v CIs exclude 0
#                    quiver_subsample=6,
#                    cmap="viridis",
#                    arrow_color="black",
#                    arrow_alpha=0.85,
#                    arrow_scale=100,
#                    stipple_size=1,
#                    stipple_face="black",
#                    stipple_edge="black",
#                    projection=None,
#                    coast=True,
#                    title=None):
#     """
#     Plot mean wind for scope with optional hour selection and significance.
#     Pass hour=12 (or any) safely. If neither hour nor avg_over_hour set, uses first hour.
#     """
#     # Required variable base names
#     base_vars = [f"{scope}_u_mean", f"{scope}_v_mean",
#                  f"{scope}_u_std", f"{scope}_v_std",
#                  f"{scope}_uv_cov",
#                  f"{scope}_u_ci_low", f"{scope}_u_ci_high",
#                  f"{scope}_v_ci_low", f"{scope}_v_ci_high"]
#     for var in base_vars:
#         if var not in summary:
#             raise ValueError(f"Missing required variable '{var}' in summary dataset.")

#     def pick(da):
#         if "hour" not in da.dims:
#             return da
#         if avg_over_hour:
#             return da.mean("hour")
#         # select a single hour
#         use_hour = hour
#         if use_hour is None:
#             # fallback: first available hour value
#             use_hour = int(da.coords["hour"].values[0])
#         # drop=True removes hour dimension -> 2D
#         return da.sel(hour=use_hour, drop=True)

#     # Extract & reduce
#     u      = pick(summary[f"{scope}_u_mean"])
#     v      = pick(summary[f"{scope}_v_mean"])
#     u_std  = pick(summary[f"{scope}_u_std"])
#     v_std  = pick(summary[f"{scope}_v_std"])
#     uv_cov = pick(summary[f"{scope}_uv_cov"])
#     u_lo   = pick(summary[f"{scope}_u_ci_low"])
#     u_hi   = pick(summary[f"{scope}_u_ci_high"])
#     v_lo   = pick(summary[f"{scope}_v_ci_low"])
#     v_hi   = pick(summary[f"{scope}_v_ci_high"])

#     # Ensure all now 2D (lat, lon)
#     for da_name in ["u","v","u_std","v_std","uv_cov","u_lo","u_hi","v_lo","v_hi"]:
#         da = locals()[da_name]
#         if set(da.dims) != set(["lat","lon"]):
#             raise ValueError(f"{da_name} is not 2D (lat, lon); got dims {da.dims}")

#     mag = np.hypot(u, v)
#     lat = u.coords["lat"]
#     lon = u.coords["lon"]

#     # Significance mask
#     if show_significance:
#         u_var = u_std**2
#         v_var = v_std**2
#         det = u_var * v_var - uv_cov**2
#         det_safe = xr.where(det > 0, det, np.nan)
#         # Hotelling T²
#         T2 = (v_var * u**2 - 2 * uv_cov * u * v + u_var * v**2) / det_safe
#         p_hot = 1 - chi2.cdf(T2, 2)
#         invalid = xr.ufuncs.isnan(T2)
#         comp_u = (u_lo > 0) | (u_hi < 0)
#         comp_v = (v_lo > 0) | (v_hi < 0)
#         comp_sig = (comp_u & comp_v) if strict_vector else (comp_u | comp_v)
#         sig_mask = ((p_hot < sig_alpha) & (~invalid)) | (invalid & comp_sig)
#     else:
#         sig_mask = xr.zeros_like(u, dtype=bool)

#     # Plot
#     have_cartopy = False
#     try:
#         import cartopy.crs as ccrs
#         have_cartopy = True
#         if projection is None:
#             projection = ccrs.PlateCarree()
#         data_crs = ccrs.PlateCarree()
#     except ImportError:
#         data_crs = None

#     if have_cartopy:
#         fig, ax = plt.subplots(figsize=(10,6), subplot_kw={"projection": projection})
#         mesh = ax.pcolormesh(lon, lat, mag, transform=data_crs, cmap=cmap)
#     else:
#         fig, ax = plt.subplots(figsize=(10,6))
#         mesh = ax.pcolormesh(lon, lat, mag, shading="auto", cmap=cmap)

#     plt.colorbar(mesh, ax=ax, pad=0.02, label=f"|{scope} mean wind| (m/s)")

#     sl_lat = slice(0, None, quiver_subsample)
#     sl_lon = slice(0, None, quiver_subsample)
#     if have_cartopy:
#         ax.quiver(lon[sl_lon], lat[sl_lat],
#                   u.isel(lon=sl_lon, lat=sl_lat),
#                   v.isel(lon=sl_lon, lat=sl_lat),
#                   transform=data_crs,
#                   color=arrow_color, alpha=arrow_alpha,
#                   scale=arrow_scale, width=0.0022, zorder=3)
#     else:
#         lon2d, lat2d = np.meshgrid(lon, lat)
#         ax.quiver(lon2d[sl_lat, sl_lon], lat2d[sl_lat, sl_lon],
#                   u.values[sl_lat, sl_lon], v.values[sl_lat, sl_lon],
#                   color=arrow_color, alpha=arrow_alpha,
#                   scale=arrow_scale, width=0.0022, zorder=3)

#     if show_significance:
#         yy, xx = np.where(sig_mask.values)
#         if len(yy):
#             if have_cartopy:
#                 ax.scatter(lon.values[xx], lat.values[yy],
#                            s=stipple_size, facecolors=stipple_face,
#                            edgecolors=stipple_edge, linewidths=0.3,
#                            marker="o", alpha=0.9,
#                            transform=data_crs, zorder=5)
#             else:
#                 ax.scatter(lon.values[xx], lat.values[yy],
#                            s=stipple_size, facecolors=stipple_face,
#                            edgecolors=stipple_edge, linewidths=0.3,
#                            marker="o", alpha=0.9, zorder=5)

#     if have_cartopy and coast:
#         ax.coastlines(linewidth=0.8, zorder=6)
#     if not have_cartopy:
#         ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

#     if title is None:
#         base = f"{scope.upper()} mean wind"
#         if "hour" in summary.dims:
#             if avg_over_hour:
#                 base += " (hour mean)"
#             else:
#                 use_hour = hour if hour is not None else int(summary.hour.values[0])
#                 base += f" (hour={use_hour})"
#         title = base
#     ax.set_title(title)

#     from matplotlib.lines import Line2D
#     legend_items = [Line2D([0],[0], color=arrow_color, lw=1.5, label="Vector")]
#     if show_significance:
#         legend_items.append(Line2D([0],[0], marker="o", linestyle="none",
#                                    markerfacecolor=stipple_face,
#                                    markeredgecolor=stipple_edge,
#                                    markersize=4, label="Significant"))
#     ax.legend(handles=legend_items, loc="upper right", frameon=True)

#     plt.tight_layout()
#     return fig, ax
    
# fig, ax = plot_mean_wind(summary, scope="hw", hour=12)
# fig.show()
# fig, ax = plot_mean_wind(summary, scope="bl", avg_over_hour=True)
# fig.show()

In [23]:
# ds_stats = summarize_bootstrap(hw_boot_results, bl_boot_results)

In [24]:
# %%time
# encoding = {
#     var: {"zlib": True, "complevel": 4, "shuffle": True}
#     for var in ds_stats.data_vars
#     }

# # Save heatwave bootstrap results
# ds_stats.to_netcdf("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/bootstrap_stats.nc",
#                          encoding=encoding,
#                          compute=True)